### 第一步：判断是否连续

In [12]:
import re

def _getIdx(node_name):
	### eg. '/model.0/conv/Conv' -> '0'\
	m = re.search(r'model\.(\d+)', node_name)
	idx = int(m.group(1)) 
	return idx

In [13]:
from collections import Counter
from itertools import groupby

def _checkContinuous(arr):
	### arr: [(node_name, weight_size), ...]
	continuousArr = []

	# 第一步：统计每个idx出现次数。
	count = Counter(_getIdx(node_name) for node_name, _ in arr)

    #第二步：若idx出现次数大于1，且连续出现，则作为数组加入continousArr。
	for idx, group in groupby(arr, key=lambda x: _getIdx(x[0])):
		group = list(group)
		if count[idx] > 1 and len(group) == count[idx]:
			continuousArr.append(group)
		else:
			continuousArr.extend(group)

    # 第三步：返回continuousArr
	return continuousArr

### 简单测试
arr = [('/model.0/conv/Conv', 10), ('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30), ('/model.1/conv/Conv', 40)]
print(_checkContinuous(arr))

[('/model.0/conv/Conv', 10), [('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30)], ('/model.1/conv/Conv', 40)]


### 第二步：计算百分比
#### getClosestNodes(arr, target) -> closestNodes

arr: [(node_name, weight_size), ...]

In [14]:
def _reshape(obj):
    ### [(x1, y1), (x2, y2), ...] -> [ [x1, x2, ...], y1+y2+... ]
    if isinstance(obj, list):
        x, y = [], 0
        for i in obj:
            x.append(i[0])
            y += i[1]
    else:
        x, y = obj[0], obj[1]

    return [x, y]

In [15]:
def getClosestNodes(arr, target):
    arr = [tuple(p) for p in arr]       
    continuousArr = _checkContinuous(arr)
    reshapedArr = [_reshape(i) for i in continuousArr]

    closestNodes, closestParam = [], 0

    for k, (nodes, param) in enumerate(reshapedArr):
        if abs(target - closestParam - param) < abs(target - closestParam):
            closestParam += param
            closestNodes.append(nodes)
        else:
            break

    closestNodes = [y for x in closestNodes for y in (x if isinstance(x, list) else [x])]
    return closestNodes

### 简单测试
# arr = [('/model.0/conv/Conv', 10), [('/model.2/conv/Conv', 20), ('/model.2/conv/Conv', 30)], ('/model.1/conv/Conv', 40)]
# target = 70
# print(getClosestNodes(arr, target))

### 第三步：组装

getQuantizedNodes(arrs, total_param) -> NODES

arrs: {"arr_name": arr[[],[]]}
NODES: {(closestNodes):(ratio, arr)}

In [16]:
def calTargetParam(targetRatio, totalParam):
	return targetRatio * totalParam
# (1 - targetRatio) / 0.75 * totalParam

In [17]:
def addConfig(targetRatio, targetParam, actualRatio, actualParam, got, arr_name):
	config = {
		'arr_name': arr_name,
		'targetRatio': targetRatio,
		'targetParam': targetParam,
		'actualRatio': actualRatio,
		'actualParam': actualParam,		
		'got': got,
	}
	return config

In [18]:
def addQuantizedNodes(arr, arrName, targetRatio, totalParam, NODES):
	### add and config
	targetParam = calTargetParam(targetRatio, totalParam)
	got = getClosestNodes(arr, targetParam)
	NODES[tuple(got)].append((targetRatio, arrName))

	actualParam = sum(w for _, w in arr if _ in got)
	actualRatio = actualParam / totalParam

	config = addConfig(
				targetRatio=targetRatio,
				targetParam=targetParam,
				actualParam=actualParam,
				actualRatio=actualRatio,
				got=got,
				arr_name=arrName
			)
	return config, actualRatio 

In [19]:
from collections import defaultdict
import numpy as np
from itertools import product

def getQuantizedNodes(arrs, total_param, step):
	### the first item in arrs is the baseline
	NODES = defaultdict(list)

	configs = []
	baselineRatios = []

	first, *rest = arrs.items()
	for targetRatioInt in range(1, step):
		name, arr = first 
		targetRatio = targetRatioInt / step

		config, baselineRatio = addQuantizedNodes(arr, name, targetRatio, total_param, NODES)
		
		configs.append(config)
		baselineRatios.append(baselineRatio)

	for (name, arr), baselineRatio in product(rest, baselineRatios):
		config, _ = addQuantizedNodes(arr, name, baselineRatio, total_param, NODES)
		configs.append(config)
	
	import json

	with open('configs.json', 'w') as f:
		json.dump({
			'baselineRatios': baselineRatios,
			'configs': configs,
		}, f, indent=4)
		
	return NODES

### 第四步：获取量化节点

In [20]:
arr_block_layer = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.9/cv1/conv/Conv', 32768], ['/model.17/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.22/cv1/conv/Conv', 98304], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.8/cv2/conv/Conv', 98304], ['/model.8/cv1/conv/Conv', 65536], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.10/cv1/conv/Conv', 65536], ['/model.10/cv2/conv/Conv', 65536], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.13/cv1/conv/Conv', 49152], ['/model.13/cv2/conv/Conv', 24576], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/cv1/conv/Conv', 24576], ['/model.19/cv2/conv/Conv', 24576], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.6/cv2/conv/Conv', 24576], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.6/cv1/conv/Conv', 16384], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.4/cv2/conv/Conv', 12288], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.4/cv1/conv/Conv', 4096], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.1/conv/Conv', 4608], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.16/cv1/conv/Conv', 16384], ['/model.16/cv2/conv/Conv', 6144], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.2/cv2/conv/Conv', 3072], ['/model.2/m.0/cv2/conv/Conv', 1152], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.16/m.0/cv1/conv/Conv', 512]]
arr_block = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.9/cv1/conv/Conv', 32768], ['/model.17/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/cv1/conv/Conv', 98304], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.8/cv2/conv/Conv', 98304], ['/model.8/cv1/conv/Conv', 65536], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.10/cv1/conv/Conv', 65536], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.10/cv2/conv/Conv', 65536], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/cv1/conv/Conv', 49152], ['/model.13/cv2/conv/Conv', 24576], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.19/cv1/conv/Conv', 24576], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/cv2/conv/Conv', 24576], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.6/cv2/conv/Conv', 24576], ['/model.6/cv1/conv/Conv', 16384], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.4/cv2/conv/Conv', 12288], ['/model.4/cv1/conv/Conv', 4096], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.1/conv/Conv', 4608], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/cv1/conv/Conv', 16384], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.16/cv2/conv/Conv', 6144], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.16/m.0/cv1/conv/Conv', 512], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.2/cv2/conv/Conv', 3072], ['/model.2/m.0/cv2/conv/Conv', 1152]]
arr_layer = [['/model.0/conv/Conv', 432], ['/model.7/conv/Conv', 294912], ['/model.5/conv/Conv', 147456], ['/model.20/conv/Conv', 147456], ['/model.9/cv2/conv/Conv', 131072], ['/model.22/cv1/conv/Conv', 98304], ['/model.8/cv2/conv/Conv', 98304], ['/model.22/cv2/conv/Conv', 98304], ['/model.22/m.0/m.0.0/cv2/conv/Conv', 73728], ['/model.22/m.0/m.0.0/cv1/conv/Conv', 73728], ['/model.23/one2one_cv4.2/one2one_cv4.2.0/conv/Conv', 73728], ['/model.10/cv1/conv/Conv', 65536], ['/model.8/cv1/conv/Conv', 65536], ['/model.10/cv2/conv/Conv', 65536], ['/model.13/cv1/conv/Conv', 49152], ['/model.8/m.0/m/m.0/cv1/conv/Conv', 36864], ['/model.23/proto/cv2/conv/Conv', 36864], ['/model.23/proto/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv2/conv/Conv', 36864], ['/model.17/conv/Conv', 36864], ['/model.8/m.0/m/m.1/cv1/conv/Conv', 36864], ['/model.8/m.0/m/m.0/cv2/conv/Conv', 36864], ['/model.23/proto/feat_fuse/conv/Conv', 36864], ['/model.23/one2one_cv4.1/one2one_cv4.1.0/conv/Conv', 36864], ['/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv', 36864], ['/model.3/conv/Conv', 36864], ['/model.10/m/m.0/ffn/ffn.1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/attn/qkv/conv/Conv', 32768], ['/model.10/m/m.0/attn/qkv/conv/Conv', 32768], ['/model.9/cv1/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', 32768], ['/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', 32768], ['/model.10/m/m.0/ffn/ffn.0/conv/Conv', 32768], ['/model.19/cv1/conv/Conv', 24576], ['/model.13/cv2/conv/Conv', 24576], ['/model.6/cv2/conv/Conv', 24576], ['/model.19/cv2/conv/Conv', 24576], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv', 20480], ['/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv', 18432], ['/model.23/one2one_cv4.0/one2one_cv4.0.0/conv/Conv', 18432], ['/model.6/cv1/conv/Conv', 16384], ['/model.10/m/m.0/attn/proj/conv/Conv', 16384], ['/model.16/cv1/conv/Conv', 16384], ['/model.8/m.0/cv3/conv/Conv', 16384], ['/model.22/m.0/m.0.1/attn/proj/conv/Conv', 16384], ['/model.23/proto/feat_refine.1/conv/Conv', 16384], ['/model.4/cv2/conv/Conv', 12288], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv', 10240], ['/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv1/conv/Conv', 9216], ['/model.19/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv2/conv/Conv', 9216], ['/model.6/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.19/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.13/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.6/m.0/m/m.1/cv1/conv/Conv', 9216], ['/model.13/m.0/m/m.0/cv2/conv/Conv', 9216], ['/model.23/one2one_cv4.1/one2one_cv4.1.1/conv/Conv', 9216], ['/model.23/one2one_cv4.0/one2one_cv4.0.1/conv/Conv', 9216], ['/model.23/one2one_cv4.2/one2one_cv4.2.1/conv/Conv', 9216], ['/model.8/m.0/cv2/conv/Conv', 8192], ['/model.8/m.0/cv1/conv/Conv', 8192], ['/model.23/proto/feat_refine.0/conv/Conv', 8192], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv', 6400], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv', 6400], ['/model.16/cv2/conv/Conv', 6144], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv', 5120], ['/model.4/m.0/cv2/conv/Conv', 4608], ['/model.1/conv/Conv', 4608], ['/model.4/m.0/cv1/conv/Conv', 4608], ['/model.19/m.0/cv3/conv/Conv', 4096], ['/model.6/m.0/cv3/conv/Conv', 4096], ['/model.4/cv1/conv/Conv', 4096], ['/model.13/m.0/cv3/conv/Conv', 4096], ['/model.2/cv2/conv/Conv', 3072], ['/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv2/conv/Conv', 2304], ['/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv', 2304], ['/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv', 2304], ['/model.16/m.0/m/m.1/cv1/conv/Conv', 2304], ['/model.16/m.0/m/m.0/cv2/conv/Conv', 2304], ['/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv', 2304], ['/model.22/m.0/m.0.1/attn/pe/conv/Conv', 1152], ['/model.23/proto/cv3/conv/Conv', 2048], ['/model.13/m.0/cv2/conv/Conv', 2048], ['/model.6/m.0/cv2/conv/Conv', 2048], ['/model.19/m.0/cv2/conv/Conv', 2048], ['/model.13/m.0/cv1/conv/Conv', 2048], ['/model.6/m.0/cv1/conv/Conv', 2048], ['/model.19/m.0/cv1/conv/Conv', 2048], ['/model.2/m.0/cv2/conv/Conv', 1152], ['/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv', 1152], ['/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv', 720], ['/model.10/m/m.0/attn/pe/conv/Conv', 1152], ['/model.2/m.0/cv1/conv/Conv', 1152], ['/model.2/cv1/conv/Conv', 1024], ['/model.16/m.0/cv3/conv/Conv', 1024], ['/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv', 720], ['/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv', 576], ['/model.16/m.0/cv2/conv/Conv', 512], ['/model.16/m.0/cv1/conv/Conv', 512]]
total_param = 2674656 #2765036
arrs = {'arr_block': arr_block, 'arr_layer': arr_layer, 'arr_block_layer': arr_block_layer}

In [21]:
NODES = getQuantizedNodes(arrs, total_param, step=10)

In [ ]:
print(NODES[1])

defaultdict(<class 'list'>, {('/model.0/conv/Conv', '/model.7/conv/Conv'): [(0.1, 'arr_block'), (0.11042317217616023, 'arr_layer'), (0.11042317217616023, 'arr_block_layer')], ('/model.0/conv/Conv', '/model.7/conv/Conv', '/model.5/conv/Conv', '/model.20/conv/Conv'): [(0.2, 'arr_block'), (0.22068482825454938, 'arr_layer'), (0.22068482825454938, 'arr_block_layer')], ('/model.0/conv/Conv', '/model.7/conv/Conv', '/model.5/conv/Conv', '/model.20/conv/Conv', '/model.9/cv2/conv/Conv', '/model.9/cv1/conv/Conv', '/model.17/conv/Conv'): [(0.3, 'arr_block'), (0.29572401086345307, 'arr_block_layer')], ('/model.0/conv/Conv', '/model.7/conv/Conv', '/model.5/conv/Conv', '/model.20/conv/Conv', '/model.9/cv2/conv/Conv', '/model.9/cv1/conv/Conv', '/model.17/conv/Conv', '/model.3/conv/Conv', '/model.22/cv2/conv/Conv', '/model.22/cv1/conv/Conv', '/model.22/m.0/m.0.1/ffn/ffn.1/conv/Conv', '/model.22/m.0/m.0.1/ffn/ffn.0/conv/Conv', '/model.22/m.0/m.0.1/attn/proj/conv/Conv', '/model.22/m.0/m.0.1/attn/pe/conv/

### 第五步：量化

In [23]:
import INT8

model = "yolo26n-seg.onnx"
data = "coco.yaml"

for nodes, cfgs in NODES.items():
	nodes_list = list(nodes)
	ratio, name = cfgs[0]
	
	quantizer = INT8.StaticQuantizer(model, data, nodes_list, f"r{ratio:.1f}_{name}", print_original=False, print_quantized=True)
	q_model = str(quantizer.static_quantize())

[INFO] Feeding batch, shape = (1, 3, 640, 640)
[INFO] Feeding batch, shape = (1, 3, 640, 640)


KeyboardInterrupt: 